# 00 Preprocess
Fixes the known data quirks once and writes clean tables to `data/processed/clean/`. The fixes live in `shared/preprocess.py`; this notebook runs them and checks each one.

In [ ]:
from shared import preprocess as pp
from shared.io import load
from shared.config import PROCESSED
import pandas as pd
import sys
sys.path.insert(0, '.')

CLEAN = PROCESSED / 'clean'
CLEAN.mkdir(exist_ok=True)

## Combine tracking
Quirks 1, 2, 4, 5: rebuild step distance, signed acceleration, one name per drill, text NA to null.

In [2]:
ct = load('combine_tracking')
ct = pp.na_to_null(ct)
ct = pp.harmonize_drills(ct)
ct = pp.add_kinematics(ct, ['event_id'])
ct.shape

(463189, 16)

In [3]:
ct.head()

,draft_year,event_id,nfl_id,entity_type,time,drill_type,drill_name,attempt,x,y,s,a,dis,dir,step_dist,a_signed
323930,2024,20240229100000_1709237434916_000,57216,PLAYER,2024-02-29T20:10:34.500,FORTY_YARD_DASH,FORTY_YARD_DASH,1,10.900000,-1.433333,0.586667,3.335603,0.001111,63.725788,NaN,NaN
323931,2024,20240229100000_1709237434916_000,57216,PLAYER,2024-02-29T20:10:34.600,FORTY_YARD_DASH,FORTY_YARD_DASH,1,11.000000,-1.433333,0.943333,3.692252,0.010000,70.346176,0.100000,3.566667
323932,2024,20240229100000_1709237434916_000,57216,PLAYER,2024-02-29T20:10:34.700,FORTY_YARD_DASH,FORTY_YARD_DASH,1,11.100000,-1.433333,1.310000,4.038059,0.010000,73.594005,0.100000,3.666667
323933,2024,20240229100000_1709237434916_000,57216,PLAYER,2024-02-29T20:10:34.800,FORTY_YARD_DASH,FORTY_YARD_DASH,1,11.266667,-1.333333,1.793333,4.309581,0.029523,71.800835,0.194365,4.833333
323934,2024,20240229100000_1709237434916_000,57216,PLAYER,2024-02-29T20:10:34.900,FORTY_YARD_DASH,FORTY_YARD_DASH,1,11.466667,-1.266667,2.236667,4.540093,0.040246,73.995960,0.210819,4.433333


Check 1: rebuilt step distance should match speed x 0.1 s (ratio near 1). The original `dis` should not.

In [4]:
chk = ct.dropna(subset=['step_dist'])
chk = chk[chk['s'] > 1]
pd.Series({'step_dist / (s*0.1)': (chk['step_dist'] / (chk['s'] * 0.1)).median(),
           'dis / (s*0.1)':       (chk['dis'] / (chk['s'] * 0.1)).median()})

step_dist / (s*0.1)    0.999473
dis / (s*0.1)          0.481645
dtype: float64

Check 4: renamed drills now appear in every class they were run in.

In [5]:
(ct.drop_duplicates('event_id')
   .query("drill_name in @pp.DRILL_RENAMES.values()")
   .groupby(['drill_name', 'draft_year']).size().unstack(fill_value=0))

draft_year,2023,2024,2025
drill_name,,,
LINE_DRILL,50,95,72
OVER_SHOULDER_ADJUST,50,43,54
PASS_PRO_MIRROR_DRILL,37,46,38
SPEED_OUT_ROUTE_LEFT,46,37,41


In [6]:
ct.to_parquet(CLEAN / 'combine_tracking.parquet', index=False)

## Player play
Quirk 5. Kept as the source for play direction.

In [7]:
plays = pp.na_to_null(load('player_play'))
plays.to_parquet(CLEAN / 'player_play.parquet', index=False)

In [8]:
plays.head()

,game_id,play_id,team_abbr,nfl_id,lined_up_position,play_description,quarter,down,yards_to_go,possession_team,...,sack,tackle,assist,caused_forced_fumble,recovered_fumble,tackle_for_loss,time_to_pressure,time_to_qb_hurry,coverage_assignment,coverage_assignment_at_snap
0,2025111610,2279,BAL,57183,T,(8:42) (Shotgun) L.Jackson pass incomplete dee...,3,1,10,BAL,...,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN
1,2026011701,1626,SF,58248,INTERIOR_LINE,(3:48) (Shotgun) K.Walker left end to SEA 28 f...,2,2,7,SEA,...,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN
2,2025101900,2838,LAR,57140,EDGE,(7:01) C.Van Lanen reported in as eligible. T...,3,2,2,JAX,...,0.0,NaN,NaN,NaN,None,None,NaN,NaN,NO_ASSIGNMENT,NO_ASSIGNMENT
3,2025111300,2628,NYJ,58209,T,(4:05) I.Williams left end to NE 22 for 4 yard...,3,1,10,NYJ,...,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN
4,2025121406,2653,NYG,57191,OLB,"(10:08) (No Huddle, Shotgun) M.Mariota scrambl...",3,2,10,WAS,...,0.0,1.0,0.0,0.0,None,False,NaN,NaN,HOOK_CURL_LEFT,CURL_FLAT_LEFT


## Game tracking, one season at a time
Quirks 3 and 6: signed acceleration from `s` for every season, and plays flipped so the offense moves toward +x.

In [ ]:
for yr in (2023, 2024, 2025):
    g = pp.na_to_null(load(f'game_tracking_{yr}'))
    g = pp.add_kinematics(g, ['game_id', 'play_id', 'nfl_id'])
    g = pp.standardize_direction(g, plays)
    g.to_parquet(CLEAN / f'game_tracking_{yr}.parquet', index=False)
    print(yr, len(g), 'raw a min', round(
        g['a'].min(), 2), '| a_signed min', round(g['a_signed'].min(), 2))

2023 3665443 raw a min 0.0 | a_signed min -77.4
2024 7827944 raw a min 0.0 | a_signed min -83.2
2025 11160903 raw a min -14.4 | a_signed min -81.6


In [10]:
g.head()  # last season processed (2025)

,game_id,play_id,nfl_id,time,x,y,s,a,dis,o,dir,event,step_dist,a_signed,play_direction
0,2025073151,80,57796,2025-08-01T00:13:30.200,80.93,28.39,0.02,0.01,0.00,97.19,286.83,NaN,NaN,NaN,right
1,2025073151,80,57796,2025-08-01T00:13:30.300,80.93,28.39,0.02,0.00,0.00,97.19,293.01,NaN,0.00,0.0,right
2,2025073151,80,57796,2025-08-01T00:13:30.400,80.94,28.39,0.01,0.00,0.01,98.28,342.57,NaN,0.01,-0.1,right
3,2025073151,80,57796,2025-08-01T00:13:30.500,80.95,28.39,0.01,0.01,0.01,99.03,48.90,NaN,0.01,0.0,right
4,2025073151,80,57796,2025-08-01T00:13:30.600,80.95,28.39,0.01,0.01,0.00,99.03,48.65,NaN,0.00,0.0,right


Check 3: raw `a` has negatives only in 2025; `a_signed` has negatives in every season, so the seasons now match.

Check 6 (direction) gets added when we pick the first play-level feature.